In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import scipy.stats as stats
from datetime import datetime

# ── Load data (Phase 1: ignore Priority column) ──────────────
df = pd.read_csv('ER_Phase2_Group_13.csv')
df = df[['Patient_ID', 'Arrival_Clock', 'Service_Required_Min']].copy()

base = datetime.strptime('08:00:00', '%H:%M:%S')

def clock_to_min(times):
    """Convert HH:MM:SS timestamps to minutes from 08:00, handling midnight rollovers."""
    result, day_offset, prev_sec = [], 0, -1
    for t in times:
        dt  = datetime.strptime(t, '%H:%M:%S')
        sec = dt.hour * 3600 + dt.minute * 60 + dt.second
        if sec < prev_sec:
            day_offset += 86400
        prev_sec = sec
        result.append((sec + day_offset - base.hour * 3600) / 60)
    return result

df['arrival_min']     = clock_to_min(df['Arrival_Clock'].tolist())
df['interarrival_min'] = df['arrival_min'].diff()

# ── Parameters ───────────────────────────────────────────────
total_time = df['arrival_min'].iloc[-1] - df['arrival_min'].iloc[0]
lam = (len(df) - 1) / total_time          # arrival rate (patients/min)
mu  = 1 / df['Service_Required_Min'].mean()  # service rate per doctor (patients/min)
s   = 5
rho = lam / (s * mu)
cv  = df['Service_Required_Min'].std() / df['Service_Required_Min'].mean()

print('=' * 55)
print('TASK 1 — PARAMETER ESTIMATION')
print('=' * 55)
print(f'Total patients         : {len(df)}')
print(f'Observation window     : {total_time:.1f} min  ({total_time/60:.1f} h)')
print(f'')
print(f'λ  (arrival rate)      = {lam:.4f} pat/min  ({lam*60:.2f} pat/h)')
print(f'1/λ (mean inter-arr.)  = {1/lam:.4f} min')
print(f'μ  (service rate/doc)  = {mu:.4f} pat/min')
print(f'1/μ (mean service)     = {1/mu:.2f} min')
print(f's  (number of doctors) = {s}')
print(f'ρ  (utilization)       = {rho:.4f}  ({rho*100:.1f}%)')
print(f'Cv (coeff. of var.)    = {cv:.4f}')

print()
print('Descriptive Statistics — Service Times:')
print(df['Service_Required_Min'].describe().round(3).to_string())
print()
print('Descriptive Statistics — Inter-Arrival Times:')
print(df['interarrival_min'].describe().round(3).to_string())

In [ ]:
matplotlib.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'figure.dpi': 150,
})

ia  = df['interarrival_min'].dropna()
svc = df['Service_Required_Min']
shape, loc_ln, scale_ln = stats.lognorm.fit(svc, floc=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# ── Left: Inter-Arrival ──────────────────────────────────────
ax = axes[0]
ax.hist(ia, bins=35, density=True, color='#4878CF', alpha=0.65,
        edgecolor='white', linewidth=0.5, label='Observed data')
x = np.linspace(0, ia.quantile(0.995), 400)
ax.plot(x, stats.expon.pdf(x, scale=1/lam), color='#C44E52', lw=2.0,
        label=fr'Exponential fit  ($\lambda$={lam:.4f} min$^{{-1}}$)')
ax.set_title('Inter-Arrival Times', fontweight='bold')
ax.set_xlabel('Time (minutes)')
ax.set_ylabel('Probability Density')
ax.set_xlim(left=0)
ax.legend(framealpha=0.9)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines[['top', 'right']].set_visible(False)

# ── Right: Service Times ─────────────────────────────────────
ax2 = axes[1]
ax2.hist(svc, bins=35, density=True, color='#DD8452', alpha=0.65,
         edgecolor='white', linewidth=0.5, label='Observed data')
x2 = np.linspace(svc.min() * 0.5, svc.quantile(0.998), 400)
ax2.plot(x2, stats.lognorm.pdf(x2, shape, loc_ln, scale_ln), color='#C44E52', lw=2.0,
         label=fr'Lognormal fit  ($\sigma$={shape:.3f}, $\mu_{{\ell}}$={np.log(scale_ln):.3f})')
ax2.set_title('Service Times', fontweight='bold')
ax2.set_xlabel('Time (minutes)')
ax2.set_ylabel('Probability Density')
ax2.set_xlim(left=0)
ax2.legend(framealpha=0.9)
ax2.grid(axis='y', alpha=0.3, linestyle='--')
ax2.spines[['top', 'right']].set_visible(False)

fig.suptitle('Figure 1 — Empirical Distributions with Fitted Theoretical Curves\n(Group 13,  n = 1,000 patients)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('histograms.png', dpi=200, bbox_inches='tight')
plt.show()

# ── KS Tests ─────────────────────────────────────────────────
ks_ia  = stats.kstest(ia,  'expon',   args=(0, 1/lam))
ks_svc = stats.kstest(svc, 'lognorm', args=(shape, loc_ln, scale_ln))

print('=' * 60)
print('TASK 1 — GOODNESS-OF-FIT  (Kolmogorov-Smirnov Test, α=0.05)')
print('=' * 60)
print()
print('Inter-Arrival Times  vs.  Exponential Distribution')
print(f'  KS statistic : {ks_ia.statistic:.4f}')
print(f'  p-value      : {ks_ia.pvalue:.4f}')
print(f'  Decision     : {"Fail to reject H₀  →  Good fit ✓" if ks_ia.pvalue > 0.05 else "Reject H₀  →  Poor fit ✗"}')
print()
print('Service Times  vs.  Lognormal Distribution')
print(f'  KS statistic : {ks_svc.statistic:.4f}')
print(f'  p-value      : {ks_svc.pvalue:.4f}')
print(f'  Decision     : {"Fail to reject H₀  →  Good fit ✓" if ks_svc.pvalue > 0.05 else "Reject H₀  →  Poor fit ✗"}')
print()
print(f'Lognormal parameters:  σ = {shape:.4f},  μ_log = {np.log(scale_ln):.4f}')

In [ ]:
from math import factorial

# ── Erlang-C (M/M/s) ─────────────────────────────────────────
def erlang_c(s, lam, mu):
    """Returns P(wait > 0) — the Erlang-C probability."""
    r = lam / mu          # total offered load
    rho = r / s           # per-server utilization
    if rho >= 1:
        return 1.0
    # P0 denominator
    sum_terms = sum((r ** n) / factorial(n) for n in range(s))
    last_term  = (r ** s) / (factorial(s) * (1 - rho))
    P0 = 1 / (sum_terms + last_term)
    Cw = ((r ** s) / (factorial(s) * (1 - rho))) * P0
    return Cw

r   = lam / mu          # total offered load (Erlangs)
rho = lam / (s * mu)   # per-server utilization

Cw  = erlang_c(s, lam, mu)

# M/M/5 metrics
Wq_mm5 = Cw / (s * mu - lam)          # avg wait in queue (min)
Lq_mm5 = lam * Wq_mm5                 # avg queue length (patients)
W_mm5  = Wq_mm5 + 1/mu                # avg time in system (min)
L_mm5  = lam * W_mm5                  # avg number in system

# M/G/5 Kingman approximation
Wq_mg5 = Wq_mm5 * (1 + cv**2) / 2
Lq_mg5 = lam * Wq_mg5
W_mg5  = Wq_mg5 + 1/mu
L_mg5  = lam * W_mg5

print('=' * 58)
print('TASK 2 — THEORETICAL MODELING')
print('=' * 58)
print(f'Offered load  r  = λ/μ = {r:.4f} Erlangs')
print(f'Utilization   ρ  = {rho:.4f}  ({rho*100:.1f}%)')
print(f'Erlang-C prob Cw = {Cw:.4f}')
print()
print(f'{"Metric":<30} {"M/M/5":>10} {"M/G/5":>10}')
print('-' * 52)
print(f'{"Wq  (avg wait in queue, min)":<30} {Wq_mm5:>10.4f} {Wq_mg5:>10.4f}')
print(f'{"Lq  (avg queue length)":<30} {Lq_mm5:>10.4f} {Lq_mg5:>10.4f}')
print(f'{"W   (avg time in system, min)":<30} {W_mm5:>10.4f} {W_mg5:>10.4f}')
print(f'{"L   (avg number in system)":<30} {L_mm5:>10.4f} {L_mg5:>10.4f}')
print()
print(f'Kingman correction factor  = (1 + Cv²)/2 = (1 + {cv**2:.4f})/2 = {(1+cv**2)/2:.4f}')
print(f'→ M/G/5 Wq is {((Wq_mm5-Wq_mg5)/Wq_mm5)*100:.1f}% lower than M/M/5 (Cv < 1 → less variability)')

In [ ]:
# ── TASK 3: PASTA Property ───────────────────────────────────
N_DOCTORS = 5
doctor_free = [0.0] * N_DOCTORS

all_busy_time     = 0.0
all_busy_arrivals = 0
start_times  = []
finish_times = []
prev_arrival = df['arrival_min'].iloc[0]

for _, row in df.iterrows():
    t_arrive = row['arrival_min']
    svc      = row['Service_Required_Min']

    # All 5 busy at time t iff min(doctor_free) > t
    earliest_free = min(doctor_free)
    all_busy_end  = min(earliest_free, t_arrive)
    if all_busy_end > prev_arrival:
        all_busy_time += all_busy_end - prev_arrival

    # a5: all 5 busy right when patient arrives?
    if min(doctor_free) > t_arrive:
        all_busy_arrivals += 1

    # assign to earliest-free doctor (FCFS)
    idx      = doctor_free.index(min(doctor_free))
    t_start  = max(doctor_free[idx], t_arrive)
    t_finish = t_start + svc
    doctor_free[idx] = t_finish

    start_times.append(t_start)
    finish_times.append(t_finish)
    prev_arrival = t_arrive

sim_end        = max(finish_times)
total_sim_time = sim_end - df['arrival_min'].iloc[0]

p5 = all_busy_time / total_sim_time       # time average: fraction of time all 5 busy
a5 = all_busy_arrivals / len(df)          # arrival average: fraction seeing all 5 busy

df['start_min']  = start_times
df['finish_min'] = finish_times
df['wait_min']   = df['start_min'] - df['arrival_min']

print('=' * 55)
print('TASK 3 — PASTA PROPERTY')
print('=' * 55)
print(f'Total simulation time        : {total_sim_time:.1f} min')
print(f'Total all-5-busy time        : {all_busy_time:.1f} min')
print(f'Arrivals seeing all 5 busy   : {all_busy_arrivals} / {len(df)}')
print()
print(f'p5  (time average)   = {all_busy_time:.1f} / {total_sim_time:.1f} = {p5:.4f}  ({p5*100:.2f}%)')
print(f'a5  (arrival average)= {all_busy_arrivals} / {len(df)}       = {a5:.4f}  ({a5*100:.2f}%)')
print()
print(f'|p5 - a5| = {abs(p5 - a5):.4f}  → {"PASTA holds ✓" if abs(p5-a5) < 0.05 else "PASTA does NOT hold ✗"}')

In [ ]:
import numpy as np
np.random.seed(42)

N        = 1000
REPS     = 1000
mean_svc = 1/mu
WINDOW   = 100
THRESH   = 0.05

# ── Shared simulation engine ──────────────────────────────────
def simulate_queue(inter_arrivals, service_times, n_doctors=5):
    doctor_free = [0.0] * n_doctors
    arrival = 0.0
    wait_times, finish_times = [], []
    all_busy_time = 0.0
    all_busy_arrivals = 0
    prev_arrival = inter_arrivals[0]
    for ia, svc in zip(inter_arrivals, service_times):
        arrival += ia
        earliest_free = min(doctor_free)
        ab_end = min(earliest_free, arrival)
        if ab_end > prev_arrival:
            all_busy_time += ab_end - prev_arrival
        if min(doctor_free) > arrival:
            all_busy_arrivals += 1
        idx     = doctor_free.index(min(doctor_free))
        t_start = max(doctor_free[idx], arrival)
        t_end   = t_start + svc
        doctor_free[idx] = t_end
        wait_times.append(t_start - arrival)
        finish_times.append(t_end)
        prev_arrival = arrival
    total = max(finish_times) - inter_arrivals[0]
    return np.array(wait_times), all_busy_time/total, all_busy_arrivals/len(inter_arrivals)

def make_service(cv_val, n, mean):
    if cv_val < 0.01:
        return np.full(n, mean)
    elif cv_val <= 1.0:
        k = max(1, round(1/cv_val**2))
        return np.random.gamma(k, mean/k, n)
    else:   # H2 — p < 2/(Cv²+1) ensures m2 > 0
        p     = min(0.5, 2/(cv_val**2 + 1)) - 0.001
        delta = np.sqrt(2*(cv_val**2 - 1)*(1-p)/p) / 2
        m1    = mean * (1 + delta)
        m2    = mean * (1 - p*delta/(1-p))
        mask  = np.random.rand(n) < p
        return np.where(mask, np.random.exponential(m1, n),
                              np.random.exponential(m2, n))

def find_warmup(cumwq, thresh=THRESH):
    """First patient i where cumwq stays within thresh of final value for all remaining patients."""
    final = cumwq[-1]
    if final == 0: return len(cumwq)
    for i in range(WINDOW, len(cumwq)):
        if all(abs(cumwq[j] - final) / final < thresh for j in range(i, len(cumwq))):
            return i
    return len(cumwq)

# ═══════════════════════════════════════════════════════════════
# TASK 4.1 — Warm-up Period (single run per ρ)
# ═══════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 5))

print('=' * 60)
print('TASK 4.1 — WARM-UP PERIOD  (single run, N=1000)')
print(f'Criterion: within {THRESH*100:.0f}% of final Wq and stays there')
print('=' * 60)
print(f'  {"ρ":<8} {"Wq (min)":>10} {"Stabilises at patient #":>24}')
print('-' * 46)

for rho_target, col in zip([0.70, 0.80, 0.95], ['#2196F3', '#FF9800', '#F44336']):
    lam_rho = rho_target * s * mu
    ia  = np.random.exponential(1/lam_rho, N)
    svc = np.random.lognormal(np.log(scale_ln), shape, N)
    waits, _, _ = simulate_queue(ia, svc)
    cumwq = np.cumsum(waits) / np.arange(1, N+1)
    stab = find_warmup(cumwq)
    stab_str = f'~{stab}' if stab < N else '>1000 (not stabilised)'
    ax.plot(range(1, N+1), cumwq, color=col, lw=1.5,
            label=fr'$\rho$ = {rho_target}  (Wq = {waits.mean():.2f} min,  stabilises {stab_str})')
    if stab < N:
        ax.axvline(stab, color=col, lw=0.8, linestyle=':')
    print(f'  {rho_target:<8} {waits.mean():>10.4f} {stab_str:>24}')

ax.set_title('Figure 2 — Warm-up Period: Cumulative Average Wait Time', fontweight='bold')
ax.set_xlabel('Number of patients processed')
ax.set_ylabel('Cumulative average Wq (minutes)')
ax.legend(framealpha=0.9, fontsize=9)
ax.grid(alpha=0.3, linestyle='--')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('warmup.png', dpi=200, bbox_inches='tight')
plt.show()

# ═══════════════════════════════════════════════════════════════
# TASK 4.2 — Arrival Patterns (single run)
# ═══════════════════════════════════════════════════════════════
svc_base = np.random.lognormal(np.log(scale_ln), shape, N)
patterns = {
    'Poisson (Exp)': np.random.exponential(1/lam, N),
    'Erlang-2':      np.random.gamma(2, 1/(2*lam), N),
    'Erlang-3':      np.random.gamma(3, 1/(3*lam), N),
    'Uniform':       np.random.uniform(0, 2/lam, N),
}

print()
print('=' * 70)
print('TASK 4.2 — ARRIVAL PATTERNS  (single run)')
print('=' * 70)
print(f'{"Pattern":<18} {"Wq (min)":>10} {"W (min)":>10} {"p5":>8} {"a5":>8}  PASTA')
print('-' * 70)
for name, ia in patterns.items():
    waits, p5, a5 = simulate_queue(ia, svc_base.copy())
    ok = 'holds ✓' if abs(p5-a5) < 0.05 else 'FAILS ✗'
    print(f'{name:<18} {waits.mean():>10.4f} {waits.mean()+1/mu:>10.4f} {p5:>8.4f} {a5:>8.4f}  {ok}')

# ═══════════════════════════════════════════════════════════════
# TASK 4.3 — Service Variation (1000-run avg)
# ═══════════════════════════════════════════════════════════════
cv_scenarios = [
    ('Low  Cv≈0.20 (Erlang-25)',    0.20),
    ('Low  Cv≈0.33 (Erlang-9)',     0.33),
    ('Our data Cv=0.424 (Lognorm)', None),
    ('Mid  Cv=1.00 (Exponential)',  1.00),
    ('High Cv=1.50 (Hyperexp)',     1.50),
]

print()
print('=' * 72)
print('TASK 4.3 — SERVICE VARIATION  (avg of 1,000 runs)')
print('=' * 72)
print(f'{"Scenario":<38} {"Cv":>6} {"Wq sim":>9} {"Wq Kingman":>11}')
print('-' * 72)
for name, cv_val in cv_scenarios:
    wq_list = []
    for _ in range(REPS):
        ia = np.random.exponential(1/lam, N)
        if cv_val is None:
            svc  = np.random.lognormal(np.log(scale_ln), shape, N)
            cv_a = cv
        else:
            svc  = make_service(cv_val, N, mean_svc)
            cv_a = cv_val
        waits, _, _ = simulate_queue(ia, svc)
        wq_list.append(waits.mean())
    king = Wq_mm5 * (1 + cv_a**2) / 2
    print(f'{name:<38} {cv_a:>6.3f} {np.mean(wq_list):>9.4f} {king:>11.4f}')